# TP 2 — Nettoyer, imputer, visualiser : `clients_brut.csv`
**Analyse de données — Master IA / Master RSD** · Chapitre 2 : Exploration et préparation des données

*Nettoyer, imputer, visualiser : le cycle complet sur un cas concret.*

**Objectifs du TP**
1. **Nettoyer** un jeu de données brut : détecter et corriger les doublons, les incohérences et les problèmes de format → *données fiables prêtes à l'analyse*.
2. **Traiter les valeurs manquantes** : appliquer plusieurs méthodes d'imputation et comparer leurs impacts → *choisir la méthode la plus pertinente*.
3. **Visualiser** les données avec Matplotlib et Seaborn → *comprendre et communiquer*.
4. **Documenter chaque décision** (choix, justifications, résultats) dans ce notebook → *traçabilité et reproductibilité*.

| Volume | Organisation | Données | Livrable |
|---|---|---|---|
| 2 séances de 2 h (4 h au total) | Binômes · Notebook Jupyter | `clients_brut.csv` (fictif, fourni) | Notebook complété + conclusions |

**Prérequis mobilisés :** exploration du chapitre 1 (TP 1) et méthodes des sections 4 à 7 du chapitre 2.

⭐ **Continuité :** ce TP prolonge le TP 1 — même jeu de données (`clients.csv`), en version « brute » avec défauts injectés.

⚠️ **Ordre des opérations (à respecter)** : ① Dédupliquer → ② Harmoniser → ③ Typer → ④ Valider les plages. Cet ordre évite d'introduire des erreurs dans les étapes suivantes.

**Mode d'emploi** : complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Auto-correction** qui suit. Répondez aux questions dans les cellules **✍️**.

---
## Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer

sns.set_theme(style="whitegrid")   # astuce : graphiques plus lisibles
pd.set_option("display.max_columns", 20)

---
## Étape 1 — Nettoyer avec Pandas : doublons et incohérences
*Un script complet et commenté — à reproduire cellule par cellule.*

**Bonne pratique :** contrôler `shape` avant/après chaque étape — le notebook doit **raconter** le nettoyage.

### 1.0 Charger le fichier brut
Le séparateur est `;` (comme au TP 1).

In [ ]:
df = ...   # TODO : pd.read_csv(...)

print(df.shape)
df.head()

In [ ]:
df.dtypes   # observez le type de la colonne age : pourquoi n'est-elle pas numérique ?

✅ **Auto-correction**

In [ ]:
assert df.shape == (1012, 8), f"Attendu (1012, 8), obtenu {df.shape} -> vérifiez sep=';'"
print("✅ 1.0 correct")

### 1.1 Doublons : compter puis supprimer
Comptez les lignes **exactement dupliquées** avec `duplicated()`, puis supprimez-les avec `drop_duplicates(keep="first")`.

In [ ]:
n_doublons = ...   # TODO
print("Doublons détectés :", n_doublons)

df = ...   # TODO : supprimer les doublons (garder la première occurrence)
print(df.shape)

✅ **Auto-correction**

In [ ]:
assert n_doublons == 12, "12 doublons attendus"
assert df.shape == (1000, 8), "1000 lignes attendues après dédoublonnage"
assert df["id"].is_unique, "chaque id doit maintenant être unique"
print("✅ 1.1 correct")

### 1.2 Harmoniser les modalités
Affichez d'abord les effectifs de `segment`, puis remplacez les modalités incohérentes.

**Bonne pratique :** `replace()` **avant** toute analyse des modalités — sinon « a » et « A » comptent comme deux segments.

In [ ]:
df["segment"].value_counts()

In [ ]:
df["segment"] = ...   # TODO : remplacer "a" par "A" et "b" par "B"

df["segment"].value_counts()

✅ **Auto-correction**

In [ ]:
assert set(df["segment"].unique()) == {"A", "B", "C"}, "il ne doit rester que A, B, C"
assert df["segment"].value_counts()["A"] == 412, "on retrouve les 412 clients A du TP 1"
print("✅ 1.2 correct")

### 1.3 Types et plages de valeurs
1. Regardez d'abord quelles valeurs d'`age` ne sont **pas numériques** (cellule fournie).
2. Convertissez `age` en numérique avec `pd.to_numeric(..., errors="coerce")` : les valeurs illisibles deviennent `NaN`.
3. Ne gardez que les âges **réalistes** (entre 0 et 120) avec `between`.

**Bonne pratique :** `errors="coerce"` transforme les valeurs illisibles en `NaN` — traitées à l'étape suivante.

In [ ]:
age_num = pd.to_numeric(df["age"], errors="coerce")
print("Valeurs non numériques :", df.loc[age_num.isna(), "age"].tolist())
print("Valeurs hors plage    :", df.loc[~age_num.between(0, 120) & age_num.notna(), "age"].tolist())

In [ ]:
df["age"] = ...   # TODO : conversion numérique
df = ...          # TODO : garder les âges entre 0 et 120

print(df.shape)

✅ **Auto-correction**

In [ ]:
assert df.shape == (988, 8), f"Attendu (988, 8), obtenu {df.shape}"
assert df["age"].dtype == "float64", "age doit être numérique"
assert df["age"].between(18, 87).all(), "les âges restants sont ceux du TP 1 (18 à 87 ans)"
print("✅ 1.3 correct — Résultat final : (988, 8) ← 12 doublons retirés, plages d'âge validées")

✍️ **Question :** `between(0, 120)` supprime aussi les âges devenus `NaN`. Pourquoi est-ce acceptable ici ? Dans quel cas faudrait-il plutôt les **imputer** ?

…

### 1.4 Valider la plage du revenu
Au TP 1, on a repéré un client à **999 999 DA** : c'est un code d'erreur de saisie, pas un vrai revenu.
Remplacez cette valeur par `NaN` (elle sera imputée à l'étape 2).

In [ ]:
print("Clients à 999 999 :", (df["revenu"] == 999999).sum())

# TODO : remplacer 999999 par NaN dans la colonne revenu (indice : df.loc[condition, "revenu"] = np.nan)

print("Revenu max :", df["revenu"].max(), "| NaN :", df["revenu"].isna().sum())

✅ **Auto-correction**

In [ ]:
assert df["revenu"].max() < 200000, "le code 999999 doit avoir disparu"
assert df["revenu"].isna().sum() == 38, "37 NaN d'origine + 1 code d'erreur = 38"
assert df.shape == (988, 8), "on ne supprime pas la ligne : seul le revenu est faux"
print("✅ 1.4 correct")

✍️ **Décision à justifier :** pourquoi remplacer le revenu par `NaN` plutôt que **supprimer la ligne** ?

…

---
## Étape 2 — Traiter les valeurs manquantes : comparer trois méthodes
*Même jeu de données, trois stratégies d'imputation, trois résultats à discuter.*

1. **Suppression** — supprimer les lignes incomplètes (`dropna`)
2. **Médiane** — imputer avec la médiane (robuste aux outliers)
3. **k-NN** — imputer avec les *k* plus proches voisins

### 2.0 Taux de valeurs manquantes avant traitement

In [ ]:
taux = ...   # TODO : proportion de NaN par colonne, arrondie à 3 décimales
taux

✅ **Auto-correction**

In [ ]:
assert taux["revenu"] == 0.038, "≈ 3,8 % de revenus manquants"
assert taux.drop("revenu").sum() == 0, "seul le revenu a des valeurs manquantes"
print("✅ 2.0 correct")

### 2.1 Appliquer les trois méthodes
Les imputeurs de scikit-learn ne travaillent que sur des colonnes **numériques** : on utilise `num_cols` (sans `id`, qui n'a pas de sens statistique).

`fit_transform` renvoie un tableau NumPy : on le remet dans un `DataFrame` avec les **mêmes colonnes et le même index** que `df`.

In [ ]:
num_cols = ["age", "revenu", "nb_enfants", "anciennete", "nb_achats"]

# Méthode 1 : suppression
df_sup = ...   # TODO

# Méthode 2 : médiane (robuste aux outliers)
imp = ...      # TODO : SimpleImputer(...)
df_med = pd.DataFrame(..., columns=num_cols, index=df.index)   # TODO

# Méthode 3 : k plus proches voisins
knn = ...      # TODO : KNNImputer(...) avec k = 5
df_knn = pd.DataFrame(..., columns=num_cols, index=df.index)   # TODO

✅ **Auto-correction**

In [ ]:
assert len(df_sup) == 950, "988 - 38 lignes incomplètes = 950"
assert df_med.shape == df_knn.shape == (988, 5)
assert df_med.isna().sum().sum() == 0 and df_knn.isna().sum().sum() == 0
assert (df_med.loc[df["revenu"].isna(), "revenu"] == df["revenu"].median()).all(), "médiane = une seule valeur"
print("✅ 2.1 correct")

### 2.2 Comparer les trois stratégies
Construisez un tableau avec, pour chaque méthode : le nombre de **lignes conservées**, de **valeurs manquantes** (revenu), le **revenu moyen**, l'**écart-type** du revenu, et l'écart du revenu moyen par rapport au brut (**Δ vs brut**).

In [ ]:
methodes = {"Brut (référence)": df, "Suppression": df_sup, "Médiane": df_med, "k-NN (k=5)": df_knn}

comparaison = pd.DataFrame({
    "lignes":       ...,   # TODO : {nom: len(d) for nom, d in methodes.items()}
    "manquantes":   ...,   # TODO : NaN dans revenu
    "revenu_moyen": ...,   # TODO
    "ecart_type":   ...,   # TODO
}).round(0)
comparaison["delta_vs_brut"] = ...   # TODO : revenu_moyen - revenu moyen du brut
comparaison

✅ **Auto-correction**

In [ ]:
assert list(comparaison["lignes"]) == [988, 950, 988, 988]
assert list(comparaison["manquantes"]) == [38, 0, 0, 0]
assert comparaison.loc["Suppression", "delta_vs_brut"] == 0
assert comparaison.loc["Médiane", "delta_vs_brut"] < comparaison.loc["k-NN (k=5)", "delta_vs_brut"] < 0
assert comparaison.loc["Médiane", "ecart_type"] < comparaison.loc["k-NN (k=5)", "ecart_type"] < comparaison.loc["Brut (référence)", "ecart_type"]
print("✅ 2.2 correct")

### 2.3 Où tombent les valeurs imputées ?
Cellule fournie : on superpose la distribution des revenus **observés** et les 38 valeurs **imputées** par chaque méthode.

In [ ]:
manquant = df["revenu"].isna()

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["revenu"], bins=40, color="lightgray", label="observés", ax=ax)
ax.vlines(df_med.loc[manquant, "revenu"], 0, 60, color="tab:orange", lw=3, label="imputés — médiane (38 fois la même valeur)")
sns.rugplot(df_knn.loc[manquant, "revenu"], color="tab:purple", height=0.15, lw=2, label="imputés — k-NN", ax=ax)
ax.set_title("Revenu : valeurs observées vs valeurs imputées")
ax.set_xlabel("Revenu (DA)")
ax.legend()
plt.show()

✍️ **Questions**
1. Pourquoi la **suppression** donne-t-elle **exactement** le même revenu moyen que le brut ? Qu'a-t-on perdu quand même ?
2. Pourquoi l'imputation par la **médiane** fait-elle **baisser** la moyenne **et** l'écart-type ?
3. Quelle méthode retenez-vous pour la suite ? Justifiez.

1. …
2. …
3. …

### 2.4 Pour aller plus loin — évaluer vraiment l'imputation
Comparer des moyennes ne dit pas si les valeurs imputées sont **justes**. Méthode classique : **cacher** des valeurs qu'on connaît, les imputer, puis mesurer l'erreur.

1. Cachez 100 revenus **connus** (tirés au hasard, graine 0).
2. Imputez avec la médiane, le k-NN brut, et le k-NN sur données **standardisées** (`StandardScaler`, section 6 du cours).
3. Comparez l'**erreur absolue moyenne** (MAE) entre valeurs imputées et vraies valeurs.

In [ ]:
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
cache = rng.choice(df.index[df["revenu"].notna()], size=100, replace=False)
vrais = df.loc[cache, "revenu"]

test = df[num_cols].copy()
test.loc[cache, "revenu"] = np.nan

def mae(rempli):
    """Erreur absolue moyenne sur les 100 revenus cachés."""
    return (rempli.loc[cache, "revenu"] - vrais).abs().mean()

# TODO : imputer `test` avec SimpleImputer(median) et KNNImputer(5), calculer mae(...) pour chacun
# TODO : k-NN standardisé : sc.fit_transform -> KNNImputer -> sc.inverse_transform

✍️ **Votre analyse :** quelle méthode se trompe le moins ? Pourquoi la standardisation change-t-elle le résultat du k-NN ?

…

⭐ **Bonus :** ajoutez le `segment` (encodé en one-hot avec `pd.get_dummies`) aux variables du k-NN standardisé, et testez aussi une **médiane par segment** (`groupby(...).transform("median")`). Que concluez-vous ?

### 2.5 Choix final et sauvegarde
On retient le **k-NN** : on remplace le revenu de `df` par celui de `df_knn`, puis on sauvegarde `clients_nettoye.csv` (réutilisé à l'étape 3).

In [ ]:
df_nettoye = df.copy()
df_nettoye["revenu"] = ...   # TODO : revenu imputé par k-NN, arrondi à la centaine (round(-2))

df_nettoye.to_csv("clients_nettoye.csv", sep=";", index=False, encoding="utf-8")
df_nettoye.shape

✅ **Auto-correction**

In [ ]:
assert df_nettoye.shape == (988, 8) and df_nettoye.isna().sum().sum() == 0
assert (df_nettoye["revenu"] % 100 == 0).all(), "revenus arrondis à la centaine"
print("✅ 2.5 correct — clients_nettoye.csv sauvegardé")

---
## Étape 3 — Visualiser avec Matplotlib et Seaborn
*Trois graphiques suffisent pour l'essentiel du diagnostic.*

| Graphique | Rôle |
|---|---|
| 1. **Histogramme** | forme de la distribution, asymétrie, outliers |
| 2. **Boxplot** | comparer le revenu entre segments (médiane, IQR, valeurs aberrantes) |
| 3. **Heatmap** | corrélations linéaires ; \|r\| > 0,8 → redondance potentielle |

Sur `df_nettoye`, tracez les trois graphiques **côte à côte** (`plt.subplots(1, 3, figsize=(18, 4))`), avec titres et étiquettes d'axes.

⚠️ `sns.distplot` est déprécié → utiliser `sns.histplot` (ou `ax.hist`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Histogramme : forme et outliers
# TODO : axes[0].hist(...) du revenu, bins=30 ; titre "Distribution du revenu", axes "Revenu (DA)" / "Effectif"

# 2. Boxplot : comparaison par segment
# TODO : sns.boxplot(data=..., x="segment", y="revenu", ax=axes[1], ...) ; titre "Revenu par segment"

# 3. Heatmap de corrélation (variables numériques, sans id)
# TODO : sns.heatmap(df_nettoye[num_cols].corr(), annot=True, cmap="Blues", fmt=".2f", ax=axes[2], ...)

plt.tight_layout()
plt.show()

✍️ **Un constat par graphique**
1. **Histogramme** : forme de la distribution ? asymétrie ? valeurs extrêmes ?
2. **Boxplot** : quel segment a le revenu le plus élevé ? les écarts sont-ils nets ?
3. **Heatmap** : quelle est la plus forte corrélation ? Y a-t-il une redondance (|r| > 0,8) ?

1. …
2. …
3. …

---
## Étape 4 — Documenter chaque décision
*Livrable : notebook complété + conclusions. Traçabilité et reproductibilité.*

✍️ **Journal des décisions** — complétez le tableau (une ligne par décision).

| # | Problème détecté | Décision | Justification | Lignes avant → après |
|---|---|---|---|---|
| 1 | … | … | … | … → … |
| 2 | … | … | … | … → … |
| 3 | … | … | … | … → … |
| 4 | … | … | … | … → … |
| 5 | … | … | … | … → … |

✍️ **Conclusions** (5 à 10 lignes) : qualité des données reçues, méthode d'imputation retenue, principaux enseignements des graphiques, limites de votre travail.

…

---
## Bilan

| Étape | Commande | Ce qu'on a appris |
|---|---|---|
| Doublons | `duplicated()`, `drop_duplicates()` | compter **avant** de supprimer |
| Modalités | `replace()`, `value_counts()` | harmoniser **avant** d'analyser |
| Types | `pd.to_numeric(errors="coerce")` | l'illisible devient NaN, traité ensuite |
| Plages | `between()` | valider les valeurs réalistes |
| Manquants | `dropna()`, `SimpleImputer`, `KNNImputer` | comparer les méthodes, pas seulement les moyennes |
| Visualisation | `hist`, `sns.boxplot`, `sns.heatmap` | distribution, groupes, corrélations |

Ordre à respecter : **Dédupliquer → Harmoniser → Typer → Valider les plages → Imputer → Visualiser**.